# DSA 2050A — Week 9 practical GROUP lab
## Model selection and defensible evaluation | Python + GitHub

**Work in your existing course groups.** Build one collaborative project and submit one shared GitHub repository link per group. List all member names, IDs and contributions in `README.md`.

**Case:** A fictional Kenyan internet provider wants to identify customers at risk of churn.
**Your group’s task:** Compare models fairly, investigate failures, document your decision, and evaluate the final test only after freezing the choice.

**Important:** Data is **synthetic for teaching**. `cancellation_confirmed` is known only *after* the outcome and is a deliberate leakage trap. Do not model using it.

**Before starting:** Open the `DSA2050_Week9_GitHub_Lab` folder in VS Code, select your Python environment, then run cells **from top to bottom**. Install packages using `python -m pip install -r requirements.txt` from the VS Code Terminal.

### Step 1 — Confirm environment and locate the datasets
**Click path:** VS Code → File → Open Folder → project folder → `notebooks/Week9_Model_Selection_Lab.ipynb` → **Select Kernel** (top right) → Python environment → **Run** each cell.

**Group discussion:** Why must the final test not be consulted before choosing a model? Add your explanation below.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate, cross_val_predict, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score, accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
from sklearn.inspection import permutation_importance
import sklearn

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
assert (DATA / 'churn_development.csv').exists(), 'Open the project folder in VS Code.'
print('Project:', ROOT.resolve())
print('Python:', sys.version.split()[0], 'scikit-learn:', sklearn.__version__)

### Step 2 — Open **development data only**, inspect, and remove leakage
Do **not** read `churn_final_test.csv` yet. Your prediction-time features must be available *before cancellation*.

In [ ]:
dev = pd.read_csv(DATA / 'churn_development.csv')
print('Shape:', dev.shape)
print('Churn rate:', round(dev['churn'].mean(), 3))
print('Missing values:\n', dev.isna().sum())
display(dev.head())

TARGET = 'churn'
LEAKAGE_OR_IDS = ['churn', 'customer_id', 'cancellation_confirmed']
X_dev = dev.drop(columns=LEAKAGE_OR_IDS)
y_dev = dev[TARGET].astype(int)
print('Allowed features:', X_dev.columns.tolist())
assert 'cancellation_confirmed' not in X_dev.columns
assert 'customer_id' not in X_dev.columns

**Write:** Identify (a) the target, (b) the unit of analysis, (c) the leakage column, and (d) why customer ID should not be used as a predictor.

*Your response:* ...

### Step 3 — Create leakage-safe preprocessing pipelines
**Analogy:** In each practice exam (CV fold), students should learn imputation values and scaling from the *practice questions* alone—not from the hidden answer sheet. The `Pipeline` re-fits preprocessing within each fold.

In [ ]:
num_cols = X_dev.select_dtypes(include='number').columns.tolist()
cat_cols = X_dev.select_dtypes(exclude='number').columns.tolist()

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
category_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, num_cols),
    ('cat', category_pipe, cat_cols)
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Numeric:', num_cols)
print('Categorical:', cat_cols)
print('Same fixed CV:', cv)

### Step 4 — Establish a naive baseline
A **DummyClassifier** predicts based on a simple rule. It tells you what success without a real model looks like.

**Predict first:** Will a majority-class model have high accuracy if most customers stay, even when it never catches churn?

In [ ]:
dummy_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', DummyClassifier(strategy='most_frequent'))
])
metrics = {'f1': 'f1', 'precision': 'precision', 'recall': 'recall'}
baseline_scores = cross_validate(dummy_pipe, X_dev, y_dev, cv=cv, scoring=metrics)
print('Baseline fold F1:', np.round(baseline_scores['test_f1'], 3))
print('Baseline mean F1:', round(baseline_scores['test_f1'].mean(), 3))

### Step 5 — Compare two candidates using identical CV folds
Logistic regression is usually stable and interpretable. A decision tree learns if/then rules but can overfit. For each, search **three understandable settings**, not hundreds.

**Primary selection metric: F1** (balances precision and recall). Report precision and recall as secondary metrics.

**Do not switch the primary metric after seeing the winner.**

In [ ]:
candidates = {
    'Logistic regression': (
        LogisticRegression(max_iter=2000, random_state=42),
        {'model__C': [0.1, 1, 10]}
    ),
    'Decision tree': (
        DecisionTreeClassifier(random_state=42),
        {'model__max_depth': [3, 5, 8]}
    )
}
searches = {}
logs = []
for name, (model, grid) in candidates.items():
    pipeline = Pipeline([('prep', preprocessor), ('model', model)])
    search = GridSearchCV(
        pipeline, grid, cv=cv, scoring=metrics,
        refit='f1', n_jobs=1, return_train_score=True
    )
    search.fit(X_dev, y_dev)
    searches[name] = search
    results = pd.DataFrame(search.cv_results_)
    for _, r in results.iterrows():
        row = {
            'model': name, 'settings': str(r['params']),
            'mean_f1': float(r['mean_test_f1']),
            'std_f1': float(r['std_test_f1']),
            'mean_precision': float(r['mean_test_precision']),
            'mean_recall': float(r['mean_test_recall'])
        }
        for i in range(5):
            row[f'fold_{i+1}_f1'] = float(r[f'split{i}_test_f1'])
        logs.append(row)

experiment_log = pd.DataFrame(logs).sort_values('mean_f1', ascending=False)
experiment_log.to_csv(REPORTS / 'experiment_log.csv', index=False)
display(experiment_log.round(3))
print('Saved:', REPORTS / 'experiment_log.csv')

### Step 6 — Examine score variation and select a candidate
Two models may have similar mean F1 but different fold stability. Larger `std_f1` means the validation score changed more across folds.

**Group task:** Compare the best logistic regression and best tree, then choose one. You **may** override the automatic highest-F1 default if you provide a defensible reason. Preserve the selected model's tuned hyperparameters.

In [ ]:
best_by_model = (experiment_log.sort_values('mean_f1', ascending=False)
                 .drop_duplicates(subset='model').reset_index(drop=True))
display(best_by_model[['model','mean_f1','std_f1','mean_recall','mean_precision']].round(3))

# Change only if your explanation supports the different choice.
CHOSEN_MODEL = best_by_model.iloc[0]['model']
assert CHOSEN_MODEL in searches
chosen_search = searches[CHOSEN_MODEL]
chosen_pipeline = chosen_search.best_estimator_
print('Chosen:', CHOSEN_MODEL)
print('Settings:', chosen_search.best_params_)
print('Development CV F1:', round(chosen_search.best_score_, 3))

### Step 7 — Out-of-fold errors and subgroup analysis
`cross_val_predict` makes each development-row prediction from a model trained on **other folds**. Use these to inspect false positives (flagged churn that did not happen) and false negatives (missed churn).

**Caution:** Out-of-fold diagnostics use the same development dataset involved in selection; do not call them an untouched final-test estimate.

In [ ]:
oof_pred = cross_val_predict(chosen_pipeline, X_dev, y_dev, cv=cv, method='predict')
print(classification_report(y_dev, oof_pred, zero_division=0))
cm = confusion_matrix(y_dev, oof_pred, labels=[0,1])
ConfusionMatrixDisplay(cm, display_labels=['Stay','Churn']).plot(cmap='Blues')
plt.title('Development out-of-fold confusion matrix')
plt.tight_layout()
plt.savefig(REPORTS / 'development_confusion_matrix.png', dpi=150)
plt.show()

errors = dev[['customer_id','customer_segment','plan_type','churn']].copy()
errors['prediction'] = oof_pred
errors['error_type'] = np.select([
    (errors.churn==1)&(errors.prediction==0),
    (errors.churn==0)&(errors.prediction==1)],
    ['False negative','False positive'],default='Correct')
print(errors.error_type.value_counts())
display(errors.query("error_type != 'Correct'").head(8))

**Subgroup task:** Is recall worse for new customers than established customers? Report subgroup sample size, number of actual churners, recall, and interpret cautiously.

In [ ]:
group_rows = []
for group, part in errors.groupby('customer_segment'):
    true = part['churn'].to_numpy()
    pred = part['prediction'].to_numpy()
    group_rows.append({
        'segment': group, 'n_customers': len(part),
        'n_actual_churn': int(true.sum()),
        'recall': recall_score(true,pred,zero_division=0),
        'precision': precision_score(true,pred,zero_division=0),
        'false_negatives': int(((true==1)&(pred==0)).sum())
    })
subgroup = pd.DataFrame(group_rows)
subgroup.to_csv(REPORTS/'subgroup_metrics.csv',index=False)
display(subgroup.round(3))

### Step 8 — Permutation importance on development validation data
Hold out a **small validation slice from the development data** for the importance demo, fit on the remaining development rows, and ask how F1 changes when one feature is shuffled. **Importance does not prove causality.**

In [ ]:
X_inner, X_validation, y_inner, y_validation = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=2050, stratify=y_dev
)
interpret_model = chosen_pipeline
interpret_model.fit(X_inner,y_inner)
importance = permutation_importance(
    interpret_model,X_validation,y_validation,
    scoring='f1', n_repeats=5,random_state=42,n_jobs=1
)
importance_table = pd.DataFrame({
    'feature': X_dev.columns,
    'importance_mean':importance.importances_mean,
    'importance_std':importance.importances_std
}).sort_values('importance_mean',ascending=False)
importance_table.to_csv(REPORTS/'permutation_importance.csv',index=False)
display(importance_table.round(3))
ax = importance_table.sort_values('importance_mean').plot.barh(
    x='feature',y='importance_mean',legend=False,figsize=(8,5))
ax.set_xlabel('Mean decrease in validation F1 after shuffling')
plt.tight_layout()
plt.savefig(REPORTS/'permutation_importance.png',dpi=150)
plt.show()

**Write:** Name the two most relied-upon features and two limitations of permutation importance (examples: correlated features, small sample, not causal).

### Step 9 — Freeze your choice BEFORE the final test
Complete the **100+ character** rationale in the next cell. It should reference the metric, cross-validation mean and variation, error behaviour, subgroup results, and complexity. The code saves a written selection record before the final-test cell.

**Do not run the final step until the rationale is complete.**

In [ ]:
RATIONALE = """REPLACE THIS TEXT with your specific evidence-based reason for selecting the model. Include mean F1, spread, false negatives, subgroup performance, and model complexity."""
assert not RATIONALE.startswith('REPLACE THIS TEXT'), 'Write your own rationale first.'
assert len(RATIONALE.strip()) >= 100, 'Write at least 100 characters.'
freeze = {
    'selected_model': CHOSEN_MODEL,
    'best_params': chosen_search.best_params_,
    'primary_metric': 'f1',
    'cv_mean_f1': float(chosen_search.best_score_),
    'rationale': RATIONALE.strip(),
    'random_seed': 42,
    'scikit_learn_version': sklearn.__version__
}
(REPORTS/'frozen_selection.json').write_text(json.dumps(freeze,indent=2),encoding='utf-8')
print('Selection frozen in reports/frozen_selection.json')

### Step 10 — ONE final evaluation: only now read the test CSV
The code refuses to recalculate final performance if `reports/final_metrics.json` already exists. **Never change a selected model after observing its final result and then claim that the same test is unbiased.**

In [ ]:
final_path = REPORTS/'final_metrics.json'
assert (REPORTS/'frozen_selection.json').exists(), 'Freeze selection before testing.'
if final_path.exists():
    print('Final test already evaluated. Reading the existing saved metrics (NOT rerunning).')
    print(final_path.read_text(encoding='utf-8'))
else:
    final_test = pd.read_csv(DATA/'churn_final_test.csv')  # first and only use
    X_test = final_test.drop(columns=LEAKAGE_OR_IDS)
    y_test = final_test[TARGET].astype(int)
    final_model = chosen_search.best_estimator_
    final_model.fit(X_dev,y_dev)  # refit on entire development dataset
    final_pred = final_model.predict(X_test)
    final_metrics = {
        'model':CHOSEN_MODEL,
        'n_test':int(len(y_test)),
        'cv_mean_f1':round(float(chosen_search.best_score_),4),
        'test_accuracy':round(float(accuracy_score(y_test,final_pred)),4),
        'test_f1':round(float(f1_score(y_test,final_pred,zero_division=0)),4),
        'test_precision':round(float(precision_score(y_test,final_pred,zero_division=0)),4),
        'test_recall':round(float(recall_score(y_test,final_pred,zero_division=0)),4),
        'confusion_matrix_0_1':confusion_matrix(y_test,final_pred,labels=[0,1]).tolist()
    }
    final_path.write_text(json.dumps(final_metrics,indent=2),encoding='utf-8')
    print(json.dumps(final_metrics,indent=2))
print('Done. Complete README and submit your GitHub repository link.')

## Group written reflection (complete before GitHub submission)
1. Which model was selected and why was it defensible? Include mean F1 and fold standard deviation.
2. What did false negatives mean for this provider? Which subgroup struggled most?
3. Which feature was highest in permutation importance, and why does that **not** imply causation?
4. Compare final-test F1 with mean CV F1. Did your model generalize reasonably?
5. State **two limitations** of this synthetic dataset and one step you would take before deploying a real churn model.

Put these answers in your project `README.md` or a short `reports/model_selection_rationale.md`. **Do not just leave “...” in the notebook.**